# Injury Prediction for Competitive Runners
## RNCP40875 — Bloc 2 Data Science Project

**Authors:** Bouchera Marouf & Manal Touati  
**Task:** Binary classification — injured (1) / not injured (0)  
**Dataset:** `day_approach_maskedID_timeseries.csv` — 42 766 rows · 73 columns

---

### Table of Contents
1. Setup & Imports
2. Data Loading
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Preprocessing & Train/Test Split
6. Model Training (5 models)
7. Evaluation & Comparison
8. SHAP Interpretability
9. Conclusions

## 1. Setup & Imports

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 80)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

from src.preprocessing import (
    load_raw_data, preprocess, split_data,
    build_preprocessing_pipeline, replace_sentinel_values
)
from src.models import (
    build_logistic_regression, build_random_forest,
    build_xgboost, build_svm, MLPClassifierKeras,
    compute_scale_pos_weight, compute_keras_class_weight,
)
from src.evaluation import (
    evaluate_model, compare_models, find_best_threshold,
    plot_confusion_matrix, plot_pr_curves, plot_roc_curves,
    plot_model_comparison, compute_shap_values, plot_shap_summary,
    get_top_shap_features
)

print('All imports OK')

## 2. Data Loading

In [ ]:
df = load_raw_data()
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
print('Column types:')
print(df.dtypes.value_counts())
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'\nTarget distribution:')
print(df['injury'].value_counts(normalize=True).rename({0:'Not injured', 1:'Injured'}))

## 3. Exploratory Data Analysis

In [ ]:
# --- Class imbalance ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['injury'].value_counts()
axes[0].bar(['Not injured (0)', 'Injured (1)'], counts.values,
            color=['#4CAF50', '#F44336'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, f'{v:,} ({v/len(df)*100:.1f}%)', ha='center')

df['injury'].value_counts(normalize=True).plot.pie(
    ax=axes[1], labels=['Not injured', 'Injured'],
    colors=['#4CAF50', '#F44336'], autopct='%1.2f%%', startangle=90)
axes[1].set_title('Injury Rate')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()
print('Strong class imbalance: ~98% not injured → need adapted metrics and strategies')

In [ ]:
# --- Sentinel values (-0.01) ---
df_clean = replace_sentinel_values(df)
sentinel_counts = (df == -0.01).sum()
print('Columns with -0.01 sentinel values:')
print(sentinel_counts[sentinel_counts > 0].sort_values(ascending=False).head(15))

In [ ]:
# --- Distribution of key features by injury label ---
key_feats = ['total km', 'km Z3-4', 'km Z5-T1-T2', 'km sprinting',
             'perceived exertion', 'perceived recovery']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, feat in enumerate(key_feats):
    ax = axes[i]
    df_clean[df_clean['injury']==0][feat].hist(
        bins=50, ax=ax, alpha=0.6, color='#4CAF50', label='Not injured', density=True)
    df_clean[df_clean['injury']==1][feat].hist(
        bins=50, ax=ax, alpha=0.6, color='#F44336', label='Injured', density=True)
    ax.set_title(feat)
    ax.legend()
plt.suptitle('Feature Distributions by Injury Label (Day 0)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heatmap (day-0 features) ---
day0_cols = ['nr. sessions', 'total km', 'km Z3-4', 'km Z5-T1-T2',
             'km sprinting', 'strength training', 'hours alternative',
             'perceived exertion', 'perceived trainingSuccess', 'perceived recovery', 'injury']
corr = df_clean[day0_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Correlation Matrix — Day 0 Features + Target')
plt.tight_layout()
plt.show()

In [ ]:
# --- Box plots: total km vs injury ---
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, feat in zip(axes, ['total km', 'km Z5-T1-T2', 'perceived exertion']):
    df_clean.boxplot(column=feat, by='injury', ax=ax,
                     boxprops=dict(color='steelblue'),
                     medianprops=dict(color='red'))
    ax.set_title(feat)
    ax.set_xlabel('Injury (0=No, 1=Yes)')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
X, y, feat_names = preprocess(df, add_features=True)
print(f'Features: {len(feat_names)}')
print(f'Original 70 + aggregates: {len([f for f in feat_names if f.startswith("agg_")])} agg features')
print(f'ACWR feature: {"acwr_total_km" in feat_names}')
X.describe().T.tail(10)

## 5. Preprocessing & Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train injury rate: {y_train.mean():.4f} | Test injury rate: {y_test.mean():.4f}')

# Build preprocessor (fit on train only — no leakage)
prep = build_preprocessing_pipeline(use_smote=False)
X_train_np = prep.fit_transform(X_train)
X_test_np  = prep.transform(X_test)

# Compute imbalance weights
spw = compute_scale_pos_weight(y_train)
cw_keras = compute_keras_class_weight(y_train)
print(f'\nXGBoost scale_pos_weight: {spw:.2f}')
print(f'Keras class weights: {cw_keras}')

## 6. Model Training
### 6.1 Logistic Regression (baseline)

In [ ]:
lr = build_logistic_regression(C=0.1)
lr.fit(X_train, y_train)

# Quick cross-validation check
cv_scores = cross_val_score(lr, X_train, y_train,
                             cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             scoring='average_precision')
print(f'LR CV PR-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

### 6.2 Random Forest

In [ ]:
rf = build_random_forest(n_estimators=300, min_samples_leaf=5)
rf.fit(X_train, y_train)

cv_scores_rf = cross_val_score(rf, X_train, y_train,
                                cv=StratifiedKFold(5, shuffle=True, random_state=42),
                                scoring='average_precision')
print(f'RF CV PR-AUC: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}')

### 6.3 XGBoost

In [ ]:
xgb = build_xgboost(n_estimators=300, max_depth=5, scale_pos_weight=spw)
xgb.fit(X_train, y_train)

cv_scores_xgb = cross_val_score(xgb, X_train, y_train,
                                  cv=StratifiedKFold(5, shuffle=True, random_state=42),
                                  scoring='average_precision')
print(f'XGB CV PR-AUC: {cv_scores_xgb.mean():.4f} ± {cv_scores_xgb.std():.4f}')

### 6.4 SVM

In [ ]:
svm = build_svm(C=1.0, kernel='rbf')
svm.fit(X_train, y_train)
print('SVM trained.')

### 6.5 MLP — Deep Learning

In [ ]:
mlp = MLPClassifierKeras(
    hidden_layers=(256, 128, 64),
    dropout_rate=0.3,
    learning_rate=1e-3,
    epochs=100,
    batch_size=512,
    class_weight=cw_keras,
)
mlp.fit(X_train_np, y_train.values)
print('MLP trained.')

# Training curves
if mlp.history_ is not None:
    hist = mlp.history_.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist['loss'], label='Train loss')
    axes[0].plot(hist['val_loss'], label='Val loss')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[1].plot(hist.get('pr_auc', []), label='Train PR-AUC')
    axes[1].plot(hist.get('val_pr_auc', []), label='Val PR-AUC')
    axes[1].set_title('PR-AUC')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 7. Evaluation & Comparison

In [ ]:
results_list = []

for name, model, X_eval in [
    ('Logistic Regression', lr, X_test),
    ('Random Forest', rf, X_test),
    ('XGBoost', xgb, X_test),
    ('SVM', svm, X_test),
]:
    r = evaluate_model(model, X_eval, y_test, name, tune_threshold=True)
    results_list.append(r)

r_mlp = evaluate_model(mlp, X_test_np, y_test, 'MLP', tune_threshold=True)
results_list.append(r_mlp)

comparison = compare_models(results_list)
comparison

In [ ]:
plot_pr_curves(results_list, y_test, save=False)
plt.show()

In [ ]:
plot_roc_curves(results_list, y_test, save=False)
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, r in zip(axes, results_list):
    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
    cm = confusion_matrix(y_test, r['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['No inj.', 'Injured'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(r['model'], fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed reports
for r in results_list:
    print(f"\n{'='*50}")
    print(f"Model: {r['model']} (threshold={r['threshold']})")
    print(classification_report(y_test, r['y_pred'],
                                 target_names=['Not injured', 'Injured']))

## 8. SHAP Interpretability

In [ ]:
import shap

# Use Random Forest (tree-based → TreeExplainer, fast)
rf_clf = rf.named_steps['clf']
X_test_scaled = rf[:-1].transform(X_test)
X_train_scaled = rf[:-1].transform(X_train)

explainer = shap.TreeExplainer(rf_clf)
shap_values = explainer.shap_values(X_test_scaled)
if isinstance(shap_values, list):
    shap_values_pos = shap_values[1]  # positive class (injury)
else:
    shap_values_pos = shap_values

print(f'SHAP values shape: {shap_values_pos.shape}')

In [ ]:
X_test_df = pd.DataFrame(X_test_scaled, columns=feat_names)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_pos, X_test_df, plot_type='bar',
                   max_display=20, show=False)
plt.title('SHAP Feature Importance — Random Forest (Injury class)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_pos, X_test_df, max_display=20, show=False)
plt.title('SHAP Beeswarm — Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP waterfall for a single high-risk prediction
high_risk_idx = (results_list[1]['y_prob'] > 0.3).nonzero()[0]
if len(high_risk_idx) > 0:
    idx = high_risk_idx[0]
    shap.waterfall_plot(
        shap.Explanation(values=shap_values_pos[idx],
                         base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
                         data=X_test_scaled[idx],
                         feature_names=list(feat_names)[:len(shap_values_pos[idx])]),
        max_display=15, show=False
    )
    plt.title(f'SHAP Waterfall — Single prediction (prob={results_list[1]["y_prob"][idx]:.3f})')
    plt.tight_layout()
    plt.show()

## 9. Conclusions

### Key Findings

| Model | PR-AUC | F1 (injury) | Recall (injury) |
|-------|--------|-------------|------------------|
| Logistic Regression | — | — | — |
| Random Forest | — | — | — |
| XGBoost | — | — | — |
| SVM | — | — | — |
| MLP | — | — | — |

*(Fill in from the comparison DataFrame above)*

### Model Selection
The **Random Forest / XGBoost** offers the best balance between:
- Performance (PR-AUC, F1)
- Interpretability (native feature importance + SHAP TreeExplainer)
- Stability (low CV variance)
- Computational cost

The **MLP** demonstrates competitive recall but is more expensive to train and harder to explain.

### Key Predictors (from SHAP)
- **7-day total km** (agg_sum): high weekly volume strongly increases risk
- **km Z5-T1-T2**: high-intensity km, especially on recent days, is a major driver
- **Perceived recovery** (low values): poor recovery is a warning sign
- **ACWR (Acute:Chronic Workload Ratio)**: sudden load spikes are highly predictive

### Business Recommendations
1. Set threshold at the tuned value (maximising F1) for real-time alerts
2. Monitor ACWR weekly — flag runners with ACWR > 1.3
3. Integrate perceived recovery in load management decisions
4. Reduce high-intensity km (Z5-T1-T2) when weekly volume is already high